# EDA — November 2026

One section per question, newest at the bottom: `## YYYY-MM-DD — question`, the data range it used, the code, and a **Finding** line.
Loaders and definitions live in `nba/eda.py` (read its docstring). The month's conclusions go in `2026-11_recap.md` next to this notebook.

Data: `NBA_DB_PATH` (MacBook: the restored backup) and the odds snapshots in `nba/data/odds_live/` (rsync from the mini) and Pikkit exports in `nba/data/pikkit/`.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from nba import eda

pd.set_option('display.width', 250); pd.set_option('display.max_columns', 30)
con = eda.connect()
END = '2026-11-23'   # last graded date for this month's questions
b = eda.bets(con, end=END)
print(f"{len(b):,} bets, {b.date.min()} to {b.date.max()}")

## 2026-11-24 — Overall: are we calibrated on the bets we take? Bench vs starters?
Data: bets Oct 20 – Nov 23. `z_model` < −2 = the model is overconfident in that group. `book_exp` = break-even (vig in).

In [ ]:
display(eda.summary(b, 'side'))
display(eda.summary(b, ['side', 'role']))
b['bad_day'] = b.date.isin(['2026-11-15', '2026-11-23'])
display(eda.summary(b, ['bad_day', 'side']))
b['edge_bin'] = pd.cut(b.claimed, [-1, .05, .10, .15, .20, 1])
display(eda.summary(b, ['side', 'edge_bin']))

**Finding (Nov 23 data):** overconfident on both sides (overs z −3.6, unders −4.4); wins track the book, not the model. Starter unders −$2.36k (−6.5%, 932 bets) are the loss; bench unders +4.7%, bench overs +2.4%. Unders still −0.6% without 11/15 and 11/23. Unders lose in every claimed-edge bin except > 20 pts.

## 2026-11-24 — Did whole games go wrong, or ladders?
Data: bets Oct 20 – Nov 23. Ladder = 2+ lines on the same player, side and day.

In [ ]:
g = b.groupby(['date', 'game_id']).agg(bets=('stake', 'size'), stake=('stake', 'sum'), pnl=('pnl', 'sum'), wins=('won', 'sum'),
                                       model_exp=('model_p', 'sum')).reset_index().sort_values('pnl')
display(g.head(10))
print(f"worst 10 games {g.pnl.head(10).sum():,.0f} | best 10 {g.pnl.tail(10).sum():,.0f} | per-game SD {g.pnl.std():,.0f}")
b['ladder'] = b.groupby(['player_id', 'date', 'side']).line.transform('size') >= 2
display(eda.summary(b, ['side', 'ladder']))

**Finding:** no whole-game misreads (game total-threes miss vs game P&L corr −0.16); worst and best 10 games are symmetric (−$3.1k / +$3.3k). Same-player ladders carry the correlation (+0.44). Under ladders −$1,320 on 116 bets; over ladders +$417.

## 2026-11-24 — Why did unders work last season and not this one? (Pikkit)
Data: every Pikkit export in `nba/data/pikkit/`, settled model plays. `clv` = closing implied p − taken implied p (+ = line moved toward the bet).

In [ ]:
pk = eda.pikkit_model_plays(con)
u = pk[(pk.over_under == 'Under') & pk.season.isin(['2025-26', '2026-27'])].copy()
u['bucket_px'] = pd.cut(u.amer, [-1e4, -150, -110, 110, 140, 200, 300, 1e4],
                        labels=['<=-150', '-150/-110', '-110/+110', '+110/+140', '+140/+200', '+200/+300', '+300+'])
s = lambda x: pd.Series({'bets': len(x), 'staked': x.amount.sum(), 'pl': x.pl.sum(), 'roi%': 100 * x.pl.sum() / x.amount.sum(),
                         'win': x.won.mean(), 'break_even': x.book_p.mean(), 'clv_pts': 100 * x.clv.mean()})
display(u.groupby('season').apply(s).round(3))
display(u.groupby(['season', 'bucket_px'], observed=True).apply(s).round(3))
display(u.groupby(['season', 'book']).apply(s).round(3))
display(u.groupby(['season', 'role']).apply(s).round(3))
display(u[u.amer >= 140].groupby('season').number.value_counts().unstack().fillna(0))

**Finding:** 2025-26 unders +10.3% on $91k; 2026-27 −2.8% on $41k. Last year 37% of under stake was +140 or longer, mostly Under 0.5 at theScore / Bet365 (+$4.9k, half the under profit); this year 9%, and 66% is −110 or shorter on main lines. Same books and prices (FD + DK, shorter than +140) still flipped from about +10% to −10% (~2 SE). Starter vs bench wasn't a split last year (+10.0% / +10.7%). CLV is better this year (+0.74 pts).

## 2026-11-24 — How many bets had a teammate starter out, and did those unders lose more?
Data: bets Oct 20 – Nov 23; `eda.starters_out` (box score, so diagnostic only).

In [ ]:
so = eda.starters_out(con, end=END)
bs = b.merge(so, on=['team_id', 'date'], how='left')
bs['state'] = np.select([bs.n_out.fillna(0) == 0, bs.any_first.fillna(False).astype(bool)], ['none out', 'first game out'], 'continuing absence')
print(f"bets with a teammate starter out: {(bs.n_out.fillna(0) > 0).mean():.0%}")
display(eda.summary(bs, ['side', 'state']))
display(eda.summary(bs[bs.side == 'Under'], ['role', 'state']))

**Finding:** about 40% of bets had a teammate starter out. Starter unders lost with or without one (none out −6.1% to −7.6%; continuing −6% to −10%; first game −3% to −5%): absences explain under half of the starter-under loss. Overs on teammates of an absent starter also lost (books price the bump). Across all predictions the absence effect is real (starters +0.14 threes with a starter out vs +0.06 without).

## 2026-11-24 — Overall calibration vs the books, where both price the same line
Data: odds snapshots (open ~5:30, midday ~14:00, close ~T−20), `player_threes` two-way lines, de-vigged; players who played, Oct 20 – Nov 23. Consensus = mean across 2+ books, Novig excluded. `dBrier` + = model worse.

In [ ]:
o = eda.model_vs_books(con, end=END)
display(o.groupby(['snap', 'book']).apply(eda.compare).round(3))
for snap in ('open', 'midday', 'close'):
    c = eda.consensus(o, snap)
    print(snap); display(eda.compare(c).round(3).to_frame().T); display(eda.blend(c))
c = eda.consensus(o, 'midday')
c['line_b'] = np.where(c.line >= 3.5, '3.5+', c.line.astype(str))
display(c.groupby('line_b').apply(eda.compare).round(3))
c['gap'] = pd.cut(c.model_p - c.book_p, [-1, -.10, -.05, -.02, .02, .05, .10, 1])
display(c.groupby('gap', observed=True).agg(n=('y', 'size'), model=('model_p', 'mean'), book=('book_p', 'mean'), actual=('y', 'mean')).round(3))

**Finding:** the model's P(over) at the main line is ~4–5 pts low (midday: model 0.44, books 0.48, actual 0.49) at every line, so it manufactures under edges. Every book beats it at midday and close (z ~3); at the open it is level with Pinnacle and Novig. When the model is 10+ pts below the book the book is right; 5+ above, the model is partly right (real over signal). Blend weight on the model: open ~0.24, midday ~0.1, close ~0.

## 2026-11-24 — By season-to-date 3PA per game (0–2, 2–5, 5–7, 7+)
Data: Oct 20 – Nov 23. Bucket = average 3PA in the player's earlier games this season (before tonight).

In [ ]:
oc = eda.outcomes(con, end=END).merge(eda.model_means(eda.pmf(con, end=END)), on=['player_id', 'date'])
f = lambda x: pd.Series({'player_games': len(x), '3P%': x.fg3m.sum() / x.fg3a.sum(), 'pred_3PM': x.mean_pred.mean(),
                         'actual_3PM': x.fg3m.mean(), 'z': eda.clustered_z((x.fg3m - x.mean_pred).values, x.date.values),
                         'P0_pred': x.p0.mean(), 'P0_actual': (x.fg3m == 0).mean()})
display(oc.groupby('bucket', observed=True).apply(f).round(3))
display(eda.consensus(o, 'midday').groupby('bucket', observed=True).apply(eda.compare).round(3))
display(eda.summary(b, ['side', 'bucket']))

**Finding:** 3P% rises with volume (31.9 / 35.7 / 35.8 / 36.9%). 0–2 shooters are over-predicted (0.59 vs 0.38); 2+ under-predicted. Under losses are all in 5+ 3PA shooters (5–7 −$2,178, −14.4%; 7+ −$816, −12.6%); unders on 2–5 and first games made +$1.2k. 5–7 is where the model trails the books most (+11.7 ×1000, z 2.8).